In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

block_size = 256
n_embd = 512
n_head = 8
n_layer = 6
dropout = 0.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:

checkpoint = torch.load('../model/base_with_rope.pt', map_location=device, weights_only=False)
stoi = checkpoint['stoi']
itos = checkpoint['itos']
vocab_size = checkpoint['vocab_size']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


In [3]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048):
        super().__init__()
        # Calculate the base frequencies
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2).float() / head_dim))

        # Calculate positions
        t = torch.arange(max_seq_len, dtype=torch.float32)

        # Multiply positions by frequencies
        freqs = torch.outer(t, inv_freq)

        # IMPORTANT: Register as a standard REAL tensor (Float32) to avoid DataParallel bugs
        # We will convert it to complex later in apply_rope
        self.register_buffer("freqs", freqs)

    def forward(self, x, start_pos=0):
        seq_len = x.shape[1]
        # Return frequencies shifted by the start_pos
        return self.freqs[start_pos : start_pos + seq_len]

In [4]:
# Helper function to do the actual rotation
def apply_rope(x, freqs):
    # x shape: (B, T, head_size) -> e.g., (128, 256, 64)
    x_reshaped = x.float().reshape(*x.shape[:-1], -1, 2) # Shape: (B, T, 32, 2)
    x_complex = torch.view_as_complex(x_reshaped)        # Shape: (B, T, 32)

    # --- THE FIX ---
    # Convert the real frequencies into complex numbers dynamically here
    # freqs shape: (T, 32) -> freqs_complex shape: (T, 32)
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)

    # Reshape frequencies to broadcast across the Batch dimension
    # freqs_complex shape: (T, 32) -> (1, T, 32)
    freqs_complex = freqs_complex.unsqueeze(0)
    # ---------------

    # Multiply (which perfectly rotates the vectors!)
    x_rotated = x_complex * freqs_complex                # Shape: (B, T, 32)

    # Convert back to real numbers and flatten the last dimension
    x_out = torch.view_as_real(x_rotated)                # Shape: (B, T, 32, 2)
    x_out = x_out.flatten(2)                             # Shape: (B, T, 64)

    return x_out.type_as(x)

In [5]:

class MaskedSelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.cache_k = None
        self.cache_v = None
        self.use_kv_cache = False
        self.rope = RotaryPositionalEmbedding(head_size)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # --- THE KV CACHE OFFSET ---

        start_pos = self.cache_k.shape[1] if (self.use_kv_cache and self.cache_k is not None) else 0

        # Pass the start_pos to RoPE
        freqs = self.rope(q, start_pos)
        # -------------------------------

        q = apply_rope(q, freqs)
        k = apply_rope(k, freqs)

        if self.use_kv_cache:
            if self.cache_k is not None:
                k = torch.cat([self.cache_k, k], dim=1)
                v = torch.cat([self.cache_v, v], dim=1)
            self.cache_k = k
            self.cache_v = v

        # build explicit causal mask when Q and K lengths differ
        q_len = q.shape[1]
        k_len = k.shape[1]

        if q_len == k_len:
            # for training only, not required here tho
            out = F.scaled_dot_product_attention(q, k, v,
                  attn_mask=None,
                  dropout_p=dropout if self.training else 0.0,
                  is_causal=True)
        else:
            # decode step — Q is 1 token, K/V are full sequence
            # every key position is visible to the query (it's already causal by construction)
            out = F.scaled_dot_product_attention(q, k, v,
                  attn_mask=None,
                  dropout_p=0.0,
                  is_causal=False)

        return out

    def clear_cache(self):
        self.cache_k = None
        self.cache_v = None


In [6]:

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([MaskedSelfAttention(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)



    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [7]:

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def _all_heads(self):
        for block in self.blocks:
            for head in block.sa.heads:
                yield head

    def enable_kv_cache(self):
        for head in self._all_heads():
            head.use_kv_cache = True
            head.clear_cache()

    def disable_kv_cache(self):
        for head in self._all_heads():
            head.use_kv_cache = False
            head.clear_cache()

    def forward(self, idx, targets=None, pos=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)

        x = self.blocks(tok_emb)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, use_kv_cache=False):
        if use_kv_cache:
            self.enable_kv_cache()
            # Process the initial prompt to fill the cache
            _, _ = self(idx)

        for _ in range(max_new_tokens):
            if use_kv_cache:
                idx_cond = idx[:, -1:] # Just pass the single newest token!
            else:
                idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        if use_kv_cache:
            self.disable_kv_cache()

        return idx


In [8]:
# ── Load model ─────────────────────────────────────────────────────────────
model = GPTLanguageModel().to(device)

# ADD strict=False HERE
model.load_state_dict(checkpoint['model_state_dict'], strict=False)

model.eval()

GPTLanguageModel(
  (token_embedding_table): Embedding(84, 512)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x MaskedSelfAttention(
            (key): Linear(in_features=512, out_features=64, bias=False)
            (query): Linear(in_features=512, out_features=64, bias=False)
            (value): Linear(in_features=512, out_features=64, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
            (rope): RotaryPositionalEmbedding()
          )
        )
        (proj): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=True)
          (3): Dropout(p=0.0, inplace=False)
        )
      )
      (ln1): LayerNorm((512,)

In [9]:

# ── Inference ──────────────────────────────────────────────────────────────
prompt = 'In the beginning'

In [10]:

# without KV cache
output_no_cache = decode(
    model.generate(
        torch.tensor([encode(prompt)], dtype=torch.long, device=device),
        max_new_tokens=100,
        use_kv_cache=False
    )[0].tolist()
)

In [11]:

# with KV cache
output_kv_cache = decode(
    model.generate(
        torch.tensor([encode(prompt)], dtype=torch.long, device=device),
        max_new_tokens=100,
        use_kv_cache=True
    )[0].tolist()
)

In [12]:


print("=== Without KV Cache ===")
print(output_no_cache)

=== Without KV Cache ===
In the beginning of the LORD, as the LORD hath spoken
unto the field of the LORD; and he hath been said unto them, L


In [13]:

print("\n=== With KV Cache ===")
print(output_kv_cache)


=== With KV Cache ===
In the beginning of the houses of the God of Israel hath
brought the spirit of the LORD.

15:10 And ye shall eat the


In [14]:
print(decode(
    model.generate(
        torch.tensor([encode("lord of  ")], dtype=torch.long, device=device),
        max_new_tokens=500,
        use_kv_cache=True
    )[0].tolist()
))

lord of  spoil and a man shall remain upon thee: and they
shall know that I am in the land, and the LORD shall come upon you.

20:27 And ye shall not be risen up, neither shall they redeem unto the
LORD, and say unto you; What shall ye come to yourselves with you in
the LORD, when ye shall be his servant in the land of Egypt, and nothing
whither almouring shall your soul until it be thine hand.

20:28 Then shall the breake four pearthe law shall whall whole many
declain the priestend whath day pendumble 
